# Japanese sung-lyrics ASR benchmark — Colab T4 runner

Seven systems on a frozen, speaker-disjoint Japanese utterance corpus annotated from LRCLIB:

| Tier | System | Checkpoint | Revision |
|---|---|---|---|
| baseline | `faster-whisper-base` | [Systran/faster-whisper-base](https://huggingface.co/Systran/faster-whisper-base) | `ebe41f70` |
| baseline | `faster-whisper-small` | [Systran/faster-whisper-small](https://huggingface.co/Systran/faster-whisper-small) | `536b0662` |
| baseline | `faster-whisper-medium` | [Systran/faster-whisper-medium](https://huggingface.co/Systran/faster-whisper-medium) | `08e178d4` |
| baseline | `faster-whisper-large-v3-turbo` | [deepdml/faster-whisper-large-v3-turbo-ct2](https://huggingface.co/deepdml/faster-whisper-large-v3-turbo-ct2) | `4df90f75` |
| japanese-pretrained | `kotoba-whisper-v2.0` | [kotoba-tech/kotoba-whisper-v2.0-faster](https://huggingface.co/kotoba-tech/kotoba-whisper-v2.0-faster) | `f44edd35` |
| japanese-pretrained | `anime-whisper` | [litagin/anime-whisper](https://huggingface.co/litagin/anime-whisper) | `22e2008a` |
| contribution | `whisper-small-lora-ja` | [openai/whisper-small](https://huggingface.co/openai/whisper-small) + LoRA | trained here |

**Excluded on purpose** — [`nvidia/canary-1b-v2`](https://huggingface.co/nvidia/canary-1b-v2) and [`nvidia/parakeet-tdt-0.6b-v3`](https://huggingface.co/nvidia/parakeet-tdt-0.6b-v3) each declare exactly 25 European languages and never mention Japanese, so neither can be decoded on this corpus without claiming support it does not have.

**Metrics** — CER and Character F1, no WER. Hugging Face Audio Course, [Chapter 5](https://huggingface.co/learn/audio-course/chapter5/evaluation): *"Certain languages, such as Mandarin and Japanese, have no notion of 'words', and so the WER is meaningless. Here, we revert to using the CER."*

---
### ⚠️ Before anything else: **Runtime → Change runtime type → T4 GPU → Save**
On CPU this takes hours instead of ~40 minutes. Cell 1 checks and tells you.

## 1. Confirm the GPU is actually attached

In [ ]:
import torch

if torch.cuda.is_available():
    name = torch.cuda.get_device_name(0)
    memory = torch.cuda.get_device_properties(0).total_memory / 1024**3
    print(f"GPU ready: {name} ({memory:.1f} GB), torch {torch.__version__}")
else:
    raise SystemExit(
        "No GPU. Runtime > Change runtime type > T4 GPU > Save, then rerun this cell."
    )

## 2. Upload `colab_bundle.zip`

Run **either** cell below, not both.

The bundle holds `ja_utterances.jsonl`, the cut utterance clips (Opus 48 kbit/s mono), `run_colab.py` and `ja_eval_standalone.py`. Full-length source songs are not included.

In [ ]:
# Option A: direct upload from your machine
from google.colab import files

uploaded = files.upload()  # select colab_bundle.zip
print(uploaded.keys())

In [ ]:
# Option B: from Google Drive (survives a disconnect; put the zip in MyDrive first)
from google.colab import drive

drive.mount('/content/drive')
!cp /content/drive/MyDrive/colab_bundle.zip /content/

## 3. Unzip and install dependencies

In [ ]:
!cd /content && unzip -q -o colab_bundle.zip -d bundle
!ls /content/bundle
!echo "clips:" && find /content/bundle/clips -type f | wc -l

In [ ]:
!pip install -q "faster-whisper==1.2.1" "transformers>=4.44,<5" "peft>=0.13" "accelerate>=0.34" soundfile av

## 4. Sanity check: one clip through the smallest model

Catches a broken CUDA/cuDNN setup in 30 seconds instead of 20 minutes into the real run.

In [ ]:
import json
import sys
import time

sys.path.insert(0, '/content/bundle')
from faster_whisper import WhisperModel

from run_colab import load_audio

rows = [json.loads(line) for line in open('/content/bundle/ja_utterances.jsonl', encoding='utf-8')]
probe = next(row for row in rows if row['split'] == 'test')
audio = load_audio('/content/bundle/' + probe['audio_path'])
print(f"clip {probe['clip_id']}  {probe['duration_seconds']:.1f}s  {len(audio) / 16000:.1f}s decoded")

model = WhisperModel('Systran/faster-whisper-base',
                     revision='ebe41f70d5b6dfa9166e2c581c45c9c0cfc57b66',
                     device='cuda', compute_type='float16')
started = time.perf_counter()
segments, _ = model.transcribe(audio, language='ja', task='transcribe', beam_size=5,
                               vad_filter=False, condition_on_previous_text=False)
text = ''.join(segment.text for segment in segments).strip()
elapsed = time.perf_counter() - started
print(f"decoded in {elapsed:.2f}s  (RTF {elapsed / probe['duration_seconds']:.3f}x)")
print(f"hypothesis characters: {len(text)}  reference characters: {probe['reference_characters']}")
del model

## 5. Run everything

Six pretrained systems on `validation` + `test`, then the LoRA fine-tune, then scoring. ~40 minutes on a T4.

Safe to rerun after a disconnect: prediction files are append-only and already-transcribed clips are skipped.

In [ ]:
!cd /content/bundle && python run_colab.py --stage all --device cuda

### Or run the stages one at a time

Handy when a free session is about to be reclaimed, or when you only want to redo one part.

In [ ]:
# !cd /content/bundle && python run_colab.py --stage baselines --device cuda
# !cd /content/bundle && python run_colab.py --stage baselines --systems kotoba-whisper-v2.0,anime-whisper
# !cd /content/bundle && python run_colab.py --stage finetune  --device cuda --epochs 3
# !cd /content/bundle && python run_colab.py --stage evaluate
!cd /content/bundle && python run_colab.py --help

## 6. Read the results

In [ ]:
from IPython.display import Markdown, display

for split in ('validation', 'test'):
    path = f'/content/bundle/reports/ja_{split}_benchmark_table.md'
    try:
        display(Markdown(f'### {split}\n\n' + open(path, encoding='utf-8').read()))
    except FileNotFoundError:
        print(f'{path} not written yet')

In [ ]:
import json

report = json.load(open('/content/bundle/finetune_report.json', encoding='utf-8'))
print(f"base            {report['base_model']}")
print(f"LoRA            r={report['lora_rank']} alpha={report['lora_alpha']} on {report['target_modules']}")
print(f"epochs          {report['epochs']}  (selected: {report['selected_epoch']} by {report['selection_metric']})")
print(f"training time   {report['training_minutes']:.1f} min")
print(f"train/val/test  {report['train_utterances']}/{report['validation_utterances']}/{report['test_utterances']} utterances")
print()
for entry in report['history']:
    print(f"  epoch {entry['epoch']}  loss {entry['train_loss']:.4f}  "
          f"val CER {entry['validation_cer'] * 100:.2f}%  "
          f"val CharF1 {entry['validation_character_f1'] * 100:.2f}%")

## 7. Download everything

Then paste `reports/ja_test_benchmark_table.md` into the README under `## Results`, and keep `final_metrics.json` next to it as the archival record for every published number.

In [ ]:
!cd /content/bundle && zip -qr /content/results.zip final_metrics.json finetune_report.json final_predictions reports lora_adapter
!ls -lh /content/results.zip
from google.colab import files

files.download('/content/results.zip')